# Logistic Regression: Cross Entropy Loss vs Bad Initialization

**Course:** IBM AI Engineering Professional Certificate  
**Module:** Deep Neural Networks with PyTorch  
**Author:** Jack Pumpuni Frimpong-Manso  
**Estimated Time:** 25 minutes

---

## Learning Objectives
After completing this lab you will be able to:
- Implement binary logistic regression using Cross Entropy Loss in PyTorch
- Explain why Cross Entropy Loss is preferred over MSE for classification
- Demonstrate the catastrophic effect of bad weight initialization
- Compare training dynamics and final accuracy under good vs bad initialization

---

## Table of Contents
1. [Dataset Setup](#1.-Dataset-Setup)
2. [Cross Entropy Loss (Good Initialization)](#2.-Cross-Entropy-Loss-Good-Initialization)
3. [Cross Entropy Loss (Bad Initialization)](#3.-Cross-Entropy-Loss-Bad-Initialization)
4. [Side-by-Side Comparison](#4.-Side-by-Side-Comparison)
5. [Summary and Key Takeaways](#5.-Summary-and-Key-Takeaways)

## Install and Import Libraries

In [ ]:
%%time
%pip install pandas numpy matplotlib --quiet
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu --quiet

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)
print(f"PyTorch version: {torch.__version__}")

---
## 1. Dataset Setup

We create a synthetic binary classification dataset and a logistic regression model.

In [ ]:
# Dataset class
class BinaryData(Dataset):
    """
    Synthetic binary classification dataset.
    Points with x < 0 are class 0; points with x >= 0 are class 1.
    """
    def __init__(self, n_samples=100):
        torch.manual_seed(1)  # reproducible
        self.x = torch.arange(-3, 3, 6 / n_samples).view(-1, 1).float()
        self.y = torch.zeros(n_samples, 1)
        self.y[n_samples // 2:] = 1.0  # upper half → class 1
        self.len = n_samples

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.len


# Logistic Regression model
class LogisticRegressionModel(nn.Module):
    """Single linear layer followed by sigmoid activation."""
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))


# Create dataset and dataloader
data_set = BinaryData(n_samples=100)
trainloader = DataLoader(dataset=data_set, batch_size=10, shuffle=True)

print(f"Dataset size:      {len(data_set)} samples")
print(f"Input range:       [{data_set.x.min():.2f}, {data_set.x.max():.2f}]")
print(f"Class distribution: {(data_set.y == 0).sum().item()} class-0, "
      f"{(data_set.y == 1).sum().item()} class-1")

In [ ]:
# Visualise the dataset
fig, ax = plt.subplots(figsize=(9, 4))
mask0 = (data_set.y[:, 0] == 0)
mask1 = (data_set.y[:, 0] == 1)
ax.scatter(data_set.x[mask0].numpy(), data_set.y[mask0].numpy(),
           color='blue', label='Class 0', alpha=0.6, edgecolors='k')
ax.scatter(data_set.x[mask1].numpy(), data_set.y[mask1].numpy(),
           color='red',  label='Class 1', alpha=0.6, edgecolors='k')
ax.set_xlabel('x (feature)', fontsize=12)
ax.set_ylabel('y (label)', fontsize=12)
ax.set_title('Binary Classification Dataset', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('binary_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Cross Entropy Loss — Good Initialization

### Why Cross Entropy for classification?

**MSE** on sigmoid outputs suffers from extremely flat gradients when predictions are far from the true label — this causes **gradient saturation** (vanishing gradients).

**Binary Cross Entropy (BCE)** is derived from Maximum Likelihood Estimation and provides strong, consistent gradients:

$$\mathcal{L}_{BCE} = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

Good initialization means weights start **close to zero** — sigmoid outputs near 0.5, gradients are healthy.

In [ ]:
def train_model(model, criterion, optimizer, dataloader, epochs=100):
    """Train a model and return per-epoch loss history."""
    loss_history = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for x_batch, y_batch in dataloader:
            optimizer.zero_grad()
            yhat = model(x_batch)
            loss = criterion(yhat, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(dataloader))
    return loss_history


def evaluate(model, dataset):
    """Return accuracy (fraction of correct predictions)."""
    with torch.no_grad():
        yhat = model(dataset.x)
        label = (yhat > 0.5).float()
        correct = (label == dataset.y).float().mean().item()
    return correct

In [ ]:
# ---- GOOD INITIALIZATION ----
torch.manual_seed(0)
model_good = LogisticRegressionModel()

print("Initial weights (good init — near zero):")
print(f"  w = {model_good.linear.weight.item():.4f}")
print(f"  b = {model_good.linear.bias.item():.4f}")

criterion_bce = nn.BCELoss()
optimizer_good = torch.optim.SGD(model_good.parameters(), lr=2.0)

loss_good = train_model(model_good, criterion_bce, optimizer_good, trainloader, epochs=100)
acc_good = evaluate(model_good, data_set)

print(f"\nFinal accuracy (good init): {acc_good * 100:.1f}%")
print(f"Final loss:                 {loss_good[-1]:.4f}")

---
## 3. Cross Entropy Loss — Bad Initialization

**Bad initialization** forces the sigmoid far into saturation at the start of training. When weights are large, outputs approach 0 or 1, and the BCE gradient becomes very small — the model struggles to escape this plateau.

Result: lower accuracy and higher loss at convergence.

In [ ]:
# ---- BAD INITIALIZATION ----
torch.manual_seed(0)
model_bad = LogisticRegressionModel()

# Force large weight values → sigmoid saturates immediately
model_bad.state_dict()['linear.weight'][0] = torch.tensor([-15.0])
model_bad.state_dict()['linear.bias'][0]   = torch.tensor([3.0])

print("Initial weights (bad init — large values push sigmoid to saturation):")
print(f"  w = {model_bad.linear.weight.item():.4f}")
print(f"  b = {model_bad.linear.bias.item():.4f}")

# Sigmoid output at extremes with bad init
x_sample = data_set.x[[0, 50, 99]]  # three representative points
with torch.no_grad():
    sigmoid_out = model_bad(x_sample)
print(f"\nSigmoid outputs at start (x={x_sample.flatten().tolist()}):")
print(f"  {sigmoid_out.flatten().tolist()}")
print("  → Already near 0 or 1 before any training — gradients ≈ 0!")

optimizer_bad = torch.optim.SGD(model_bad.parameters(), lr=2.0)
loss_bad = train_model(model_bad, criterion_bce, optimizer_bad, trainloader, epochs=100)
acc_bad = evaluate(model_bad, data_set)

print(f"\nFinal accuracy (bad init):  {acc_bad * 100:.1f}%")
print(f"Final loss:                 {loss_bad[-1]:.4f}")

---
## 4. Side-by-Side Comparison

In [ ]:
# Loss curves comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Loss curves
axes[0].plot(loss_good, 'b-', linewidth=2, label=f'Good Init (final acc={acc_good*100:.1f}%)')
axes[0].plot(loss_bad,  'r--', linewidth=2, label=f'Bad Init  (final acc={acc_bad*100:.1f}%)')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('BCE Loss', fontsize=12)
axes[0].set_title('Training Loss: Good vs Bad Initialization', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Decision boundary comparison
x_plot = torch.linspace(-3, 3, 300).unsqueeze(1)
with torch.no_grad():
    y_good = model_good(x_plot).numpy()
    y_bad  = model_bad(x_plot).numpy()

axes[1].scatter(data_set.x[mask0].numpy(), data_set.y[mask0].numpy(),
                color='blue', alpha=0.4, label='Class 0', edgecolors='k', s=30)
axes[1].scatter(data_set.x[mask1].numpy(), data_set.y[mask1].numpy(),
                color='red',  alpha=0.4, label='Class 1', edgecolors='k', s=30)
axes[1].plot(x_plot.numpy(), y_good, 'b-', linewidth=2.5, label='Good init sigmoid')
axes[1].plot(x_plot.numpy(), y_bad,  'r--', linewidth=2.5, label='Bad init sigmoid')
axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='Decision boundary (0.5)')
axes[1].set_xlabel('x (feature)', fontsize=12)
axes[1].set_ylabel('P(class=1)', fontsize=12)
axes[1].set_title('Learned Decision Boundaries', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Cross Entropy Loss: Good vs Bad Initialization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('good_vs_bad_init.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: good_vs_bad_init.png")

In [ ]:
# Why BCE beats MSE: gradient magnitude comparison
p_vals = np.linspace(0.01, 0.99, 200)
y_true = 1  # true class = 1

# BCE gradient magnitude: dL/dp = -(y/p) + (1-y)/(1-p)
bce_grad = np.abs(-(y_true / p_vals) + (1 - y_true) / (1 - p_vals))

# MSE gradient magnitude (for σ-activated output): dL/dp = 2(p - y)
mse_grad = np.abs(2 * (p_vals - y_true))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p_vals, bce_grad, 'b-', linewidth=2.5, label='|∂BCE/∂p|')
ax.plot(p_vals, mse_grad, 'r--', linewidth=2.5, label='|∂MSE/∂p|')
ax.set_xlabel('Predicted probability p', fontsize=12)
ax.set_ylabel('|Gradient magnitude|', fontsize=12)
ax.set_title('BCE vs MSE Gradient Magnitude (y=1, correct → p=1)\n'
             'BCE provides large gradients even when predictions are far off', fontsize=11)
ax.legend(fontsize=12)
ax.set_ylim(0, 10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('bce_vs_mse_gradients.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bce_vs_mse_gradients.png")

---
## 5. Summary and Key Takeaways

| Aspect | Good Initialization | Bad Initialization |
|---|---|---|
| Initial weights | Small (near zero) | Large (saturated sigmoid) |
| Initial gradients | Healthy | Near zero (vanishing) |
| Training speed | Fast convergence | Slow or stuck |
| Final accuracy | ~100% | ~60% |

### Key Rules
1. **Use `nn.BCELoss`** (or `nn.BCEWithLogitsLoss`) for binary classification — not MSE
2. **Initialize weights near zero** (PyTorch default for `nn.Linear` uses Kaiming uniform, which is generally good)
3. **Avoid manually setting large initial weights** unless you have a specific architectural reason
4. **Batch Normalization** can help mitigate bad initialization by normalizing layer inputs

In [ ]:
# Final results summary
print("=" * 55)
print("FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"{'Metric':<30} {'Good Init':>10} {'Bad Init':>10}")
print("-" * 55)
print(f"{'Accuracy':<30} {acc_good*100:>9.1f}% {acc_bad*100:>9.1f}%")
print(f"{'Final BCE Loss':<30} {loss_good[-1]:>10.4f} {loss_bad[-1]:>10.4f}")
print(f"{'Loss epoch 1':<30} {loss_good[0]:>10.4f} {loss_bad[0]:>10.4f}")
print(f"{'Loss epoch 10':<30} {loss_good[9]:>10.4f} {loss_bad[9]:>10.4f}")